# Phase 21 — Feature Integrity & Validation Audit

**Project:** Australian Stock Market Analytics & Return Modeling

## Objective
Audit the Phase 19/20 expanded feature table for duplicate/suffixed columns, compare duplicate-value pairs, identify leakage-risk fields, rebuild a canonical feature set, and compare baseline/Ridge/Random Forest on the fixed validation period only.

**Safeguards**
- No test-set scoring or feature selection.
- Do not silently resolve conflicting duplicate columns.
- The original Phase 19/20 files are not overwritten.
- Feature-name checks are a screening tool, not proof of point-in-time correctness.

## 1. Setup and locate project artifacts

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
EXPANDED_REL = Path("data/processed/expanded_features/asx_3m_expanded_features_targets.csv")
TRAIN_REL = Path("data/processed/model_features/asx_3m_strict_split_train.csv")
VALID_REL = Path("data/processed/model_features/asx_3m_strict_split_validation.csv")
TEST_REL = Path("data/processed/model_features/asx_3m_strict_split_test.csv")

def locate_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if all((p / r).exists() for r in [EXPANDED_REL, TRAIN_REL, VALID_REL, TEST_REL]):
            return p
    for p in start.iterdir() if start.exists() else []:
        if p.is_dir() and all((p / r).exists() for r in [EXPANDED_REL, TRAIN_REL, VALID_REL, TEST_REL]):
            return p
    return None

PROJECT_ROOT = locate_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate Phase 19 expanded table and Phase 13 split files.")

OUT = PROJECT_ROOT / "reports" / "feature_integrity_audit"
OUT.mkdir(parents=True, exist_ok=True)
CLEAN_OUT = PROJECT_ROOT / "data" / "processed" / "audited_features"
CLEAN_OUT.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

Project root: e:\Projects\Australian Stock Market Data Science Project


## 2. Load expanded table and inspect duplicate/suffixed columns

In [2]:
expanded = pd.read_csv(PROJECT_ROOT / EXPANDED_REL, low_memory=False)
expanded.columns = [str(c).strip().lower() for c in expanded.columns]
for c in ["date", "target_endpoint_date_3m"]:
    if c in expanded.columns:
        expanded[c] = pd.to_datetime(expanded[c], errors="coerce")
if "ticker" not in expanded or "date" not in expanded:
    raise ValueError("Expanded feature table must contain ticker and date.")
expanded["ticker"] = expanded["ticker"].astype(str).str.strip().str.upper()

suffix_cols = [c for c in expanded.columns if re.search(r"_(x|y)$", c)]
duplicate_names = expanded.columns[expanded.columns.duplicated()].tolist()
print("Rows:", len(expanded), "| Columns:", len(expanded.columns))
print("Duplicate column names:", duplicate_names)
print("Merge-suffixed columns:", suffix_cols)
display(pd.DataFrame({"merge_suffixed_column": suffix_cols}))

Rows: 55943 | Columns: 38
Duplicate column names: []
Merge-suffixed columns: ['momentum_21d_x', 'momentum_63d_x', 'benchmark_return_1d_x', 'benchmark_volatility_21d_x', 'benchmark_volatility_63d_x', 'momentum_21d_y', 'momentum_63d_y', 'benchmark_return_1d_y', 'benchmark_volatility_21d_y', 'benchmark_volatility_63d_y']


,merge_suffixed_column
0,momentum_21d_x
1,momentum_63d_x
2,benchmark_return_1d_x
3,benchmark_volatility_21d_x
4,benchmark_volatility_63d_x
5,momentum_21d_y
6,momentum_63d_y
7,benchmark_return_1d_y
8,benchmark_volatility_21d_y
9,benchmark_volatility_63d_y


## 3. Compare suffix pairs and classify whether they are identical

In [3]:
pair_rows = []
for c in suffix_cols:
    base = re.sub(r"_(x|y)$", "", c)
    counterpart = base + ("_y" if c.endswith("_x") else "_x")
    if counterpart in expanded.columns:
        a, b = expanded[c], expanded[counterpart]
        both = a.notna() & b.notna()
        equal = (a[both].astype(str).values == b[both].astype(str).values).mean() if both.any() else np.nan
        numeric_a = pd.to_numeric(a, errors="coerce")
        numeric_b = pd.to_numeric(b, errors="coerce")
        numeric_both = numeric_a.notna() & numeric_b.notna()
        max_abs_diff = float((numeric_a[numeric_both] - numeric_b[numeric_both]).abs().max()) if numeric_both.any() else np.nan
        pair_rows.append({
            "column_a": c, "column_b": counterpart,
            "nonmissing_overlap": int(both.sum()),
            "equal_fraction_on_overlap": float(equal) if pd.notna(equal) else np.nan,
            "max_abs_numeric_difference": max_abs_diff,
            "same_missingness": bool(a.isna().equals(b.isna()))
        })

pair_report = pd.DataFrame(pair_rows).drop_duplicates(
    subset=["column_a", "column_b"]
) if pair_rows else pd.DataFrame(columns=[
    "column_a", "column_b", "nonmissing_overlap", "equal_fraction_on_overlap",
    "max_abs_numeric_difference", "same_missingness"
])
display(pair_report)
pair_report.to_csv(OUT / "suffix_pair_comparison.csv", index=False)

,column_a,column_b,nonmissing_overlap,equal_fraction_on_overlap,max_abs_numeric_difference,same_missingness
0,momentum_21d_x,momentum_21d_y,54799,0.016971,0.680060,False
1,momentum_63d_x,momentum_63d_y,52615,0.016212,0.928572,False
2,benchmark_return_1d_x,benchmark_return_1d_y,55892,1.000000,0.000000,True
3,benchmark_volatility_21d_x,benchmark_volatility_21d_y,54808,0.000000,0.005394,False
4,benchmark_volatility_63d_x,benchmark_volatility_63d_y,52624,0.000000,0.002477,False
5,momentum_21d_y,momentum_21d_x,54799,0.016971,0.680060,False
6,momentum_63d_y,momentum_63d_x,52615,0.016212,0.928572,False
7,benchmark_return_1d_y,benchmark_return_1d_x,55892,1.000000,0.000000,True
8,benchmark_volatility_21d_y,benchmark_volatility_21d_x,54808,0.000000,0.005394,False
9,benchmark_volatility_63d_y,benchmark_volatility_63d_x,52624,0.000000,0.002477,False


## 4. Build canonical columns without silent overwrites

If both suffixed variants exist, use one canonical column only when the observed values and missingness are identical. If they conflict, preserve both source columns and flag the feature for manual review. Never overwrite the source table.

In [4]:
canonical = expanded.copy()
resolution_rows = []
drop_candidates = set()

for c in suffix_cols:
    base = re.sub(r"_(x|y)$", "", c)
    counterpart = base + ("_y" if c.endswith("_x") else "_x")
    if counterpart not in canonical.columns:
        # A lone suffixed column is renamed only if the unsuffixed name is absent.
        if base not in canonical.columns:
            canonical[base] = canonical[c]
            drop_candidates.add(c)
            resolution_rows.append({
                "canonical_feature": base, "source_columns": c,
                "resolution": "renamed lone suffixed column",
                "safe_to_collapse": True
            })
        else:
            resolution_rows.append({
                "canonical_feature": base, "source_columns": c,
                "resolution": "unsuffixed name already exists; manual review",
                "safe_to_collapse": False
            })
        continue

    a, b = canonical[c], canonical[counterpart]
    identical = a.equals(b)
    if identical:
        # Prefer an unsuffixed column if it already exists; otherwise keep _x as canonical.
        preferred = base if base in canonical.columns else base + "_x"
        canonical[base] = canonical[preferred]
        drop_candidates.update([c, counterpart])
        resolution_rows.append({
            "canonical_feature": base, "source_columns": f"{c}; {counterpart}",
            "resolution": f"collapsed identical columns; canonical source={preferred}",
            "safe_to_collapse": True
        })
    else:
        resolution_rows.append({
            "canonical_feature": base, "source_columns": f"{c}; {counterpart}",
            "resolution": "conflicting pair preserved; manual review required",
            "safe_to_collapse": False
        })

# Drop only columns proven safe to collapse and not chosen as canonical.
for c in list(drop_candidates):
    if c in canonical.columns and c not in [r.get("canonical_source") for r in resolution_rows]:
        # Keep a suffixed source if it is the canonical column selected above.
        safe = any(r["safe_to_collapse"] and r["canonical_feature"] == re.sub(r"_(x|y)$", "", c)
                   for r in resolution_rows)
        if safe and c.endswith(("_x", "_y")):
            canonical = canonical.drop(columns=c)

resolution = pd.DataFrame(resolution_rows)
display(resolution)
resolution.to_csv(OUT / "canonical_column_resolution.csv", index=False)

,canonical_feature,source_columns,resolution,safe_to_collapse
0,momentum_21d,momentum_21d_x; momentum_21d_y,conflicting pair preserved; manual review requ...,False
1,momentum_63d,momentum_63d_x; momentum_63d_y,conflicting pair preserved; manual review requ...,False
2,benchmark_return_1d,benchmark_return_1d_x; benchmark_return_1d_y,collapsed identical columns; canonical source=...,True
3,benchmark_volatility_21d,benchmark_volatility_21d_x; benchmark_volatili...,conflicting pair preserved; manual review requ...,False
4,benchmark_volatility_63d,benchmark_volatility_63d_x; benchmark_volatili...,conflicting pair preserved; manual review requ...,False
5,momentum_21d,momentum_21d_y; momentum_21d_x,conflicting pair preserved; manual review requ...,False
6,momentum_63d,momentum_63d_y; momentum_63d_x,conflicting pair preserved; manual review requ...,False
7,benchmark_return_1d,benchmark_return_1d_y; benchmark_return_1d_x,collapsed identical columns; canonical source=...,True
8,benchmark_volatility_21d,benchmark_volatility_21d_y; benchmark_volatili...,conflicting pair preserved; manual review requ...,False
9,benchmark_volatility_63d,benchmark_volatility_63d_y; benchmark_volatili...,conflicting pair preserved; manual review requ...,False


## 5. Leakage-risk screen and canonical feature inventory

In [5]:
target_or_future_patterns = re.compile(
    r"(forward_return|target_endpoint|target_price|future|next_close|label|target_horizon|target_definition)",
    flags=re.IGNORECASE
)
id_cols = {"ticker", "date", "target_endpoint_date_3m", "forward_return_3m"}
numeric_cols = [c for c in canonical.columns if pd.api.types.is_numeric_dtype(canonical[c])]
feature_candidates = [
    c for c in numeric_cols
    if c not in id_cols and not target_or_future_patterns.search(c)
    and not c.endswith(("_x", "_y"))
]
# Remove any raw price column used in target construction.
feature_candidates = [c for c in feature_candidates if c not in {"close_for_features", "target_endpoint_price_3m"}]

inventory_rows = []
for c in feature_candidates:
    inventory_rows.append({
        "feature": c,
        "missing_rate": float(canonical[c].isna().mean()),
        "unique_values": int(canonical[c].nunique(dropna=True)),
        "std": float(canonical[c].std()) if canonical[c].notna().any() else np.nan
    })
inventory = pd.DataFrame(inventory_rows).sort_values(["missing_rate", "feature"])
display(inventory)
inventory.to_csv(OUT / "canonical_feature_inventory.csv", index=False)

,feature,missing_rate,unique_values,std
0,close_for_target,0.000000,35280,33.261383
20,benchmark_return_1d,0.000912,1075,0.008295
1,momentum_1d,0.000930,49070,0.023588
10,short_term_reversal_1d,0.001859,49022,0.023613
11,benchmark_return_5d,0.004558,1071,0.018320
15,benchmark_volatility_5d,0.004558,1071,0.003928
2,momentum_5d,0.004648,51216,0.049723
16,relative_momentum_5d,0.004648,54691,0.047010
4,volatility_5d,0.004648,54479,0.014887
7,price_to_sma_21d,0.018590,54641,0.055733


## 6. Reconstruct fixed train/validation rows and verify endpoints

In [6]:
def load_split(path):
    d = pd.read_csv(path, low_memory=False)
    d.columns = [str(c).strip().lower() for c in d.columns]
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d["target_endpoint_date_3m"] = pd.to_datetime(d["target_endpoint_date_3m"], errors="coerce")
    return d

train0 = load_split(PROJECT_ROOT / TRAIN_REL)
valid0 = load_split(PROJECT_ROOT / VALID_REL)
test0 = load_split(PROJECT_ROOT / TEST_REL)
train_start, train_end = train0["date"].min(), train0["date"].max()
valid_start, valid_end = valid0["date"].min(), valid0["date"].max()
test_start = test0["date"].min()

canonical["date"] = pd.to_datetime(canonical["date"], errors="coerce")
canonical["ticker"] = canonical["ticker"].astype(str).str.strip().str.upper()
canonical["forward_return_3m"] = pd.to_numeric(canonical["forward_return_3m"], errors="coerce")

train = canonical[
    canonical["date"].between(train_start, train_end) &
    (canonical["target_endpoint_date_3m"] < valid_start)
].copy()
valid = canonical[
    canonical["date"].between(valid_start, valid_end) &
    (canonical["target_endpoint_date_3m"] < test_start)
].copy()

if train.empty or valid.empty:
    raise ValueError("Train/validation data empty after fixed boundary and endpoint checks.")
assert train["target_endpoint_date_3m"].max() < valid_start
assert valid["target_endpoint_date_3m"].max() < test_start
print("Train rows:", len(train), "| Validation rows:", len(valid))

Train rows: 40447 | Validation rows: 6084


## 7. Compare baseline and models using audited features only

In [7]:
# Restrict to features present and variable in training data. Use only features with no unresolved suffix.
usable_features = []
for c in feature_candidates:
    if c in train.columns and pd.to_numeric(train[c], errors="coerce").notna().any():
        if pd.to_numeric(train[c], errors="coerce").nunique(dropna=True) > 1:
            usable_features.append(c)

# Exclude features with unresolved duplicate variants from the model input.
unresolved_bases = set()
for _, r in resolution.iterrows():
    if not r["safe_to_collapse"]:
        unresolved_bases.add(r["canonical_feature"])
usable_features = [c for c in usable_features if c not in unresolved_bases]

if not usable_features:
    raise ValueError("No clean numeric features remain after integrity screening.")

Xtr = train[usable_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
Xva = valid[usable_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
ytr = train["forward_return_3m"].astype(float)
yva = valid["forward_return_3m"].astype(float)

models = {
    "Training mean baseline": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyRegressor(strategy="mean"))
    ]),
    "Audited Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),
    "Audited Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=250, min_samples_leaf=25, max_features=0.8,
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

results_rows, fitted = [], {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    fitted[name] = model
    p = model.predict(Xva)
    results_rows.append({
        "model": name, "n_features": len(usable_features), "n_validation": len(yva),
        "MAE": mean_absolute_error(yva, p),
        "RMSE": np.sqrt(mean_squared_error(yva, p)),
        "R2": r2_score(yva, p),
        "actual_mean": float(yva.mean()), "predicted_mean": float(np.mean(p))
    })
results = pd.DataFrame(results_rows).sort_values("MAE").reset_index(drop=True)
display(results)
results.to_csv(OUT / "audited_feature_validation_comparison.csv", index=False)

,model,n_features,n_validation,MAE,RMSE,R2,actual_mean,predicted_mean
0,Audited Ridge,21,6084,0.140900,0.195794,-0.089313,0.075098,0.030043
1,Training mean baseline,21,6084,0.141984,0.195113,-0.081749,0.075098,0.021461
2,Audited Random Forest,21,6084,0.150640,0.210510,-0.259222,0.075098,0.026467


## 8. Save audited feature table and summary

In [8]:
# Save a separate audit artifact; original Phase 19 table remains unchanged.
canonical.to_csv(CLEAN_OUT / "asx_3m_audited_features_targets.csv", index=False)

baseline_row = results[results["model"] == "Training mean baseline"].iloc[0]
best_row = results.iloc[0]
summary = {
    "phase": 21,
    "expanded_rows": int(len(expanded)),
    "source_column_count": int(len(expanded.columns)),
    "suffix_column_count": int(len(suffix_cols)),
    "suffix_pairs_compared": int(len(pair_report)),
    "feature_candidate_count": int(len(feature_candidates)),
    "usable_feature_count": int(len(usable_features)),
    "usable_features": usable_features,
    "train_rows": int(len(train)),
    "validation_rows": int(len(valid)),
    "best_validation_model": str(best_row["model"]),
    "best_validation_mae": float(best_row["MAE"]),
    "baseline_validation_mae": float(baseline_row["MAE"]),
    "best_beats_baseline": bool(best_row["MAE"] < baseline_row["MAE"]),
    "test_rows_scored": 0,
    "warning": "Column-name screening is not proof of point-in-time validity. Conflicting suffix pairs require review; test set was not scored."
}
with open(OUT / "phase21_feature_integrity_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print("\nAudit outputs:", OUT)
print("Audited feature table:", CLEAN_OUT / "asx_3m_audited_features_targets.csv")

{
  "phase": 21,
  "expanded_rows": 55943,
  "source_column_count": 38,
  "suffix_column_count": 10,
  "suffix_pairs_compared": 10,
  "feature_candidate_count": 21,
  "usable_feature_count": 21,
  "usable_features": [
    "close_for_target",
    "momentum_1d",
    "momentum_5d",
    "momentum_126d",
    "volatility_5d",
    "volatility_21d",
    "volatility_63d",
    "price_to_sma_21d",
    "price_to_sma_63d",
    "price_to_sma_126d",
    "short_term_reversal_1d",
    "benchmark_return_5d",
    "benchmark_return_21d",
    "benchmark_return_63d",
    "benchmark_return_126d",
    "benchmark_volatility_5d",
    "relative_momentum_5d",
    "relative_momentum_21d",
    "relative_momentum_63d",
    "relative_momentum_126d",
    "benchmark_return_1d"
  ],
  "train_rows": 40447,
  "validation_rows": 6084,
  "best_validation_model": "Audited Ridge",
  "best_validation_mae": 0.14089971906026702,
  "baseline_validation_mae": 0.14198443955103976,
  "best_beats_baseline": true,
  "test_rows_scored"

## 9. Completion checklist

- Review suffix pair comparisons and canonical resolution decisions.
- Confirm no unresolved duplicate feature variants entered the model.
- Review validation MAE against the baseline.
- Preserve conflicting columns until their provenance is verified.
- Do not use test metrics for feature selection or model choice.